# Crafting an AI-Powered HR Assistant: A Use Case for Nestle's HR Policy Documents
## Course-end Project 2

**Overview:**
This project implements a Retrieval-Augmented Generation (RAG) system to create a conversational chatbot. The assistant extracts information from Nestle's HR policy PDFs, converts the text into numerical vectors, and uses a vector database to retrieve relevant context for the OpenAI model to generate accurate answers.

**Key Components:**
- **PDF Extraction**: Using `PyPDF` to read policy documents.
- **Vector Embeddings**: Using OpenAI embeddings to represent text numerically.
- **Vector Store**: Using `ChromaDB` for efficient similarity search.
- **User Interface**: A professional chatbot interface built with `Gradio`.

## Phase 2: Document Ingestion & Vector Indexing
In this phase, we load the Nestle HR policy PDF, split the text into chunks, and store them in a vector database for efficient retrieval.

In [ ]:
%pip install pypdf langchain langchain-community langchain-openai chromadb gradio tiktoken langchain-text-splitters

In [ ]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
import gradio as gr

# Phase 2: Document ingestion and vector indexing

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise EnvironmentError("OPENAI_API_KEY is not set")

PDF_PATH = os.getenv(
    "PDF_PATH",
    r"C:\Dock\AI\Workspace\Simplilearn Projects\P2\the_nestle_hr_policy_pdf_2012.pdf",
)
if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}")

OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
PERSIST_DIR = "./chroma_db"

def ingest_docs():
    embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)

    # Load cached vectors when available to avoid re-embedding every run
    if os.path.exists(PERSIST_DIR) and len(os.listdir(PERSIST_DIR)) > 0:
        print("Loading existing vector database from disk...")
        vector_db = Chroma(
            persist_directory=PERSIST_DIR,
            embedding_function=embeddings,
        )
        print("Vector database loaded successfully!")
    else:
        print("No existing database found. Ingesting PDF...")
        print("Loading PDF...")
        loader = PyPDFLoader(PDF_PATH)
        documents = loader.load()

        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
        chunks = text_splitter.split_documents(documents)

        print(f"Created {len(chunks)} text chunks.")

        print("Creating vector embeddings and indexing (this may take a while)...")
        vector_db = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            persist_directory=PERSIST_DIR,
        )
        print("Vector database created and persisted!")
    return vector_db

vector_db = ingest_docs()

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Initialize the LLM
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# Define a custom prompt to ensure the AI stays within the HR policy context
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a professional Nestle HR Assistant. Use the retrieved context to answer the user's question. "
        "If the answer is not contained within the context, say that you do not have that information in the current HR policy documents. "
        "Keep the answer professional, concise, and accurate.\n\nContext:\n{context}",
    ),
    ("human", "{question}"),
])

def get_hr_answer(query):
    """
    Takes a user query, retrieves context, and returns the AI-generated answer.
    """
    docs = retriever.invoke(query)
    context = "\n\n".join(doc.page_content for doc in docs)
    messages = prompt.invoke({"context": context, "question": query})
    result = llm.invoke(messages)
    return result.content if hasattr(result, "content") else str(result)

# Test the chain
test_query = "What is the policy on maternity leave?"
print(f"Query: {test_query}")
print(f"Answer: {get_hr_answer(test_query)}")

In [ ]:
def hr_chatbot_response(message, history):
    """Interface function for Gradio."""
    return get_hr_answer(message)

demo = gr.ChatInterface(
    fn=hr_chatbot_response,
    title="Nestle HR Policy Assistant",
    description=(
        "Welcome! I am your AI-powered HR Assistant. Ask me anything about "
        "Nestle's HR policies, and I will find the answer in the official documents."
    ),
    examples=[
        "What are the working hours?",
        "How do I apply for leave?",
        "What is the policy on employee benefits?",
    ],
)

demo.launch(share=False)